# 01 — Environment Test

Verify that the LUMI environment (ROCm / CUDA), PyTorch, and HuggingFace
Transformers are correctly installed before running the heavier notebooks.

Run all cells top-to-bottom. All checks should print ✅ at the end.

In [ ]:
import sys
print(f'Python: {sys.version}')

## 1. PyTorch & GPU

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'ROCm build      : {torch.version.hip is not None}')  # True on AMD

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'GPU count       : {n_gpus}')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU {i}: {props.name} — {vram_gb:.1f} GB VRAM')
else:
    print('  WARNING: no GPU detected. Notebooks 02 and 03 will be very slow on CPU.')

In [ ]:
# Quick tensor operation on GPU
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    x = torch.randn(1024, 1024, device=device, dtype=torch.bfloat16)
    y = x @ x.T
    print(f'BF16 matmul on {device}: shape={tuple(y.shape)}, dtype={y.dtype} ✅')
else:
    device = torch.device('cpu')
    print(f'Running on CPU (no GPU found)')

## 2. Core Package Versions

In [ ]:
import importlib

packages = [
    'transformers',
    'accelerate',
    'peft',
    'datasets',
    'trl',
    'tokenizers',
    'huggingface_hub',
]

all_ok = True
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  {pkg:<20} {ver}')
    except ImportError as e:
        print(f'  {pkg:<20} MISSING ❌  ({e})')
        all_ok = False

print()
print('All packages present ✅' if all_ok else 'Some packages are missing ❌')

## 3. HuggingFace Hub Authentication

In [ ]:
import os
from huggingface_hub import whoami, HfApi

# Set token via environment variable or login interactively:
#   export HF_TOKEN="hf_..."
# or run:  huggingface-cli login

try:
    info = whoami()
    print(f'Logged in as: {info["name"]} ✅')
except Exception as e:
    print(f'Not authenticated: {e}')
    print('Set HF_TOKEN env var or run: huggingface-cli login')

## 4. Tokenizer Smoke-Test

Downloads only the tokenizer files (small) to confirm model access.

In [ ]:
import os
from pathlib import Path
from transformers import AutoTokenizer

MODEL_ID = 'Qwen/Qwen3.5-35B-A3B'

# Pin HF cache to notebooks/hf_cache regardless of Jupyter launch directory
def _find_notebooks_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'notebooks').is_dir():
            return candidate / 'notebooks'
    return Path.cwd()

HF_CACHE_DIR = _find_notebooks_dir() / 'hf_cache'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)

print(f'Downloading tokenizer for {MODEL_ID} ...')
print(f'HF cache: {HF_CACHE_DIR}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

test_text = 'Hello, LUMI supercomputer! Running Qwen3.5-35B-A3B on AMD GPUs.'
tokens = tokenizer(test_text, return_tensors='pt')

print(f'Vocab size  : {tokenizer.vocab_size:,}')
print(f'Input text  : {test_text!r}')
print(f'Token count : {tokens.input_ids.shape[1]}')
decoded = tokenizer.decode(tokens.input_ids[0], skip_special_tokens=True)
print(f'Decoded     : {decoded!r}')
print('Tokenizer OK ✅')

## 5. Memory Summary

In [ ]:
import psutil

ram = psutil.virtual_memory()
print(f'System RAM   : {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f'GPU {i} VRAM  : {total / 1024**3:.1f} GB total, {free / 1024**3:.1f} GB free')

print()
print('=== Environment check complete ===')
print('Proceed to notebook 02_inference.ipynb')